# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Properties script
**Goal:** Main script that generates the geometry dataset that acts as input for the structural analysis in grasshopper, this script can take several variables as input at generates a CSV file made up of properties that can generate this digital geometry to a CAD geometry   
**Inputs:**   
*   Define base mesh
*   Allowed movement for different vertices
*   divisions over allowed movement

**Outputs:**

*   CSV file, with sample id, vertices, coordinates and edges

# 1.1 IMPORTING AND SETTING PARAMETERS

In [1]:
import pandas as pd
import numpy as np
import random
import c11_params

# 1.2 GEOMETRY GENERATION

## Structural Attribute of vertices
script om de support points te identificeren

In [ ]:
import config
from geometry import get_corner_indices

# --- TESTEN ---
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 Grid Indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 Grid Indices: {grid_3x3}")

grid_4x4 = get_corner_indices(4, 4)
print(f"4x4 Grid Indices: {grid_4x4}")
# Verwacht: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> want 5 punten per rij

# In je loop:
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] bijv.
print(corner_values)

Systeem geladen. Project root: 2.2 - 2.4
2x2 Grid Indices: {'bottom_left': 0, 'bottom_right': 2, 'top_left': 6, 'top_right': 8}
3x3 Grid Indices: {'bottom_left': 0, 'bottom_right': 3, 'top_left': 12, 'top_right': 15}
4x4 Grid Indices: {'bottom_left': 0, 'bottom_right': 4, 'top_left': 20, 'top_right': 24}


## Calculating Vertices

In [ ]:
from geometry import bilinear_interpolate

def get_valid_shifts(divisions, edge_length):
    """Berekent de toegestane verschuivingen (verwijdert extremen)."""
    half_div = divisions // 2
    all_steps = list(range(-half_div, half_div + 1))
    valid_steps = all_steps[1:-1] # Verwijder eerste en laatste
    valid_shifts = [(step / divisions) * edge_length for step in valid_steps]
    return valid_shifts

# --- Main Function ---
def generate_full_dataset(num_samples):
    valid_shifts = get_valid_shifts(c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
    all_vertices = []

    num_nodes_x_top = c11_params.GRID_CELLS_X + 1
    num_nodes_y_top = c11_params.GRID_CELLS_Y + 1

    for sample_id in range(num_samples):
        # --- STAP 1: TOP LAYER ---
        top_layer_coords = {}
        vertex_idx = 0

        for i in range(num_nodes_y_top): # i is de rij index (y-richting)
            for j in range(num_nodes_x_top): # j is de kolom index (x-richting)
                # Basis positie
                base_x = j * c11_params.EDGE_LENGTH
                base_y = i * c11_params.EDGE_LENGTH
                base_z = 0.0

                # Structural Attribute Bepalen
                corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
                corner_values = corners.values()

                # Hier bepalen we het attribuut voor DIT punt
                if vertex_idx in corner_values:
                    attribute = "support"
                else:
                    attribute = "load"

                # Constraints
                is_x_edge = (j == 0) or (j == num_nodes_x_top - 1)
                is_y_edge = (i == 0) or (i == num_nodes_y_top - 1)
                is_corner = is_x_edge and is_y_edge

                shift_x = 0.0
                shift_y = 0.0

                if is_corner:
                    pass # Corner: Vast
                elif is_x_edge:
                    # Verticale rand (links/rechts): beweegt in Y
                    shift_y = random.choice(valid_shifts)
                elif is_y_edge:
                    # Horizontale rand (onder/boven): beweegt in X
                    shift_x = random.choice(valid_shifts)
                else:
                    # Inner: beweegt in X en Y
                    shift_x = random.choice(valid_shifts)
                    shift_y = random.choice(valid_shifts)

                final_x = base_x + shift_x
                final_y = base_y + shift_y
                final_z = base_z

                top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': final_z}

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": f"v{vertex_idx}",
                    "layer": "top",
                    "attribute": attribute,
                    "x": round(final_x, 3),
                    "y": round(final_y, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

        # --- STAP 2: LOWER LAYER ---
        # Vertex index loopt door vanaf 25
        # i loop 0..3, j loop 0..3

        for i in range(c11_params.GRID_CELLS_Y):
            for j in range(c11_params.GRID_CELLS_X):
                # We pakken de 4 hoekpunten van de top-cel
                # Rij i is 'onder', Rij i+1 is 'boven'
                p_bl = top_layer_coords[(i, j)]       # Bottom-Left
                p_br = top_layer_coords[(i, j+1)]     # Bottom-Right
                p_tl = top_layer_coords[(i+1, j)]     # Top-Left
                p_tr = top_layer_coords[(i+1, j+1)]   # Top-Right

                # Random positie in de cel (scaled 50%)
                u = random.uniform(*c11_params.SCALE_UV)
                v = random.uniform(*c11_params.SCALE_UV)

                # Interpoleren (let op volgorde parameters van mijn functie)
                # def bilinear_interpolate(p00, p10, p01, p11, u, v):
                # p00=BL, p10=BR, p01=TL, p11=TR
                lx, ly = bilinear_interpolate(p_bl, p_br, p_tl, p_tr, u, v)

                # Z positie
                base_z = -c11_params.LAYER_HEIGHT
                z_shift = random.choice(valid_shifts)
                final_z = base_z + z_shift

                all_vertices.append({
                    "sample_id": sample_id,
                    "vertex_index": f"v{vertex_idx}",
                    "layer": "bottom",
                    "attribute": "hinges",
                    "x": round(lx, 3),
                    "y": round(ly, 3),
                    "z": round(final_z, 3)
                })
                vertex_idx += 1

    return pd.DataFrame(all_vertices)

## Executing and saving

In [ ]:
import config
from geometry import generate_edges
from config import RAW_DATA_PATH

# --- EXECUTE ---
final_vertices = generate_full_dataset(c11_params.NUM_SAMPLES)
final_edges = generate_edges(c11_params.NUM_SAMPLES, c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

# --- SAVE ---
final_vertices.to_csv(RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv", index=False)
final_edges.to_csv(RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_edges.csv", index=False)

print(f"Generatie gereed voor {c11_params.NUM_SAMPLES} samples.")
print(f"Grid grootte: {c11_params.GRID}")

print("Vertices Shape:", final_vertices.shape)
print("Edges Shape:", final_edges.shape)

print(f"\nTotaal aantal regels in vertices CSV: {len(final_vertices)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_vertices.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotaal aantal regels in edges CSV: {len(final_edges)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_edges.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

# 1.5 SEARCH SPACE

In [ ]:
import pandas as pd
import json

def define_search_space(cells_x, cells_y, divisions, edge_length):
    """
    Vertaalt de geometrische constraints naar een digitaal leesbare 'Search Space'
    voor een machine learning of optimalisatie algoritme.
    """
    valid_shifts = get_valid_shifts(divisions, edge_length)
    search_space = {}

    num_nodes_x_top = cells_x + 1
    num_nodes_y_top = cells_y + 1
    vertex_idx = 0

    # Hulpfunctie om te bepalen of een punt een hoek is
    corners = get_corner_indices(cells_x, cells_y).values()

    # --- 1. TOP LAYER VARIABELEN ---
    for i in range(num_nodes_y_top):
        for j in range(num_nodes_x_top):
            v_name = f"v{vertex_idx}"

            is_x_edge = (j == 0) or (j == num_nodes_x_top - 1)
            is_y_edge = (i == 0) or (i == num_nodes_y_top - 1)
            is_corner = vertex_idx in corners

            if is_corner:
                # AI mag hier niets mee doen
                pass
            elif is_x_edge:
                # Mag alleen schuiven over de Y-as
                search_space[f"{v_name}_shift_y"] = {"type": "discrete", "options": valid_shifts}
            elif is_y_edge:
                # Mag alleen schuiven over de X-as
                search_space[f"{v_name}_shift_x"] = {"type": "discrete", "options": valid_shifts}
            else:
                # Inner node: Mag in beide richtingen schuiven
                search_space[f"{v_name}_shift_x"] = {"type": "discrete", "options": valid_shifts}
                search_space[f"{v_name}_shift_y"] = {"type": "discrete", "options": valid_shifts}

            vertex_idx += 1

    # --- 2. BOTTOM LAYER VARIABELEN ---
    for r in range(cells_y):
        for c in range(cells_x):
            v_name = f"v{vertex_idx}"

            # u en v bepalen de positie ONDER de top-cel (continue waarden tussen 0.25 en 0.75)
            search_space[f"{v_name}_u"] = {"type": "continuous", "min": c11_params.SCALE_UV[0], "max": c11_params.SCALE_UV[1]}
            search_space[f"{v_name}_v"] = {"type": "continuous", "min": c11_params.SCALE_UV[0], "max": c11_params.SCALE_UV[1]}

            # Z-shift is weer een discrete stap
            search_space[f"{v_name}_shift_z"] = {"type": "discrete", "options": valid_shifts}

            vertex_idx += 1

    return search_space

# --- UITVOEREN EN BEKIJKEN ---

# Gebruik de variabelen uit je eerdere configuratie (bijv. 1x1 grid)
ai_search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)

print(f"De Search Space bevat in totaal {len(ai_search_space)} onafhankelijke variabelen (knoppen waar de AI aan mag draaien).")
print("\nVoorbeeld van hoe de computer dit ziet:")

# Toon de eerste 5 variabelen mooi geformatteerd
for var_name, bounds in list(ai_search_space.items()):
    print(f"- {var_name}: {bounds}")

In [9]:
import os

# Sla de dictionary op als een JSON bestand
json_path = 'search_space.json'
with open(json_path, 'w') as f:
    json.dump(ai_search_space, f, indent=4) # indent=4 maakt het mooi leesbaar

print(f"✅ Search Space succesvol opgeslagen als '{json_path}'")

✅ Search Space succesvol opgeslagen als 'search_space.json'
